# ABR Car — Ingestão da Landing para a Bronze

Este notebook realiza a ingestão das sete abas do arquivo Excel da ABR Car, armazenado na camada Landing, para tabelas Delta na camada Bronze.

## Responsabilidades do notebook

- Validar a existência do arquivo de origem;
- Confirmar a presença das sete abas obrigatórias;
- Ler e inspecionar os dados com Pandas e `openpyxl`;
- Converter os DataFrames Pandas para DataFrames Spark;
- Adicionar metadados técnicos de rastreabilidade;
- Preservar snapshots semanais do arquivo;
- Gravar as tabelas Delta na camada Bronze;
- Garantir idempotência por meio do `_batch_id`;
- Validar schemas, metadados e contagens.

## Limites desta etapa

A camada Bronze deve permanecer próxima aos dados recebidos da origem. Portanto, este notebook não realizará deduplicação, correções de negócio, relacionamentos, cálculos de KPIs ou transformações destinadas às camadas Silver e Gold.

## 1. Preparação do ambiente

O Spark não possui leitura nativa de arquivos Excel nesta configuração. Por isso, utilizaremos Pandas com o mecanismo `openpyxl` para acessar as abas do arquivo.

A versão da dependência é fixada para evitar diferenças de comportamento entre execuções.

In [0]:
%pip install openpyxl==3.1.5

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import openpyxl

print(f"Versão do openpyxl: {openpyxl.__version__}")

Versão do openpyxl: 3.1.5


## 2. Parâmetros da ingestão

Esta seção centraliza os caminhos, nomes e identificadores utilizados no processamento.

A data do lote representa a versão semanal recebida da loja. Ela é definida explicitamente para que uma reexecução posterior continue utilizando o mesmo `_batch_id`, sem depender da data ou do horário em que o notebook for executado.

In [0]:
from pathlib import Path

# Identificacao do ambiente

CATALOG = "abr_car_dev"
BRONZE_SCHEMA = "bronze"

# Arquivo corrente disponibilizado pela loja
SOURCE_FILE_NAME = "PLANILHA_GERAL_ABR_CAR.xlsx"
SOURCE_FILE_PATH = (
    f"/Volumes/{CATALOG}/landing/source_files/{SOURCE_FILE_NAME}"
)

# Diretorio destinado ao historico dos arquivos recebidos
SNAPSHOT_DIRECTORY = (
    f"/Volumes/{CATALOG}/landing/source_files/snapshots"
)

# Data de referencia deste lote semanal
BATCH_DATE = "2026-08-23"
BATCH_DATE_COMPACT = BATCH_DATE.replace("-", "")

# Identificação tecnica do lote
BATCH_ID = f"abr_car_{BATCH_DATE_COMPACT}"

# Nome e caminho do snapshot
SNAPSHOT_FILE_NAME = (
    f"PLANILHA_GERAL_ABR_CAR_{BATCH_DATE_COMPACT}.xlsx"
)

SNAPSHOT_FILE_PATH = (
    f"{SNAPSHOT_DIRECTORY}/{SNAPSHOT_FILE_NAME}"
)

print(f"Arquivo atual: {SOURCE_FILE_PATH}")
print(f"Snapshot: {SNAPSHOT_FILE_PATH}")
print(f"Batch ID: {BATCH_ID}")

Arquivo atual: /Volumes/abr_car_dev/landing/source_files/PLANILHA_GERAL_ABR_CAR.xlsx
Snapshot: /Volumes/abr_car_dev/landing/source_files/snapshots/PLANILHA_GERAL_ABR_CAR_20260823.xlsx
Batch ID: abr_car_20260823


## 3. Validação do arquivo de origem

Antes de iniciar a leitura, o processo confirma que o Excel existe no caminho esperado.

Essa validação permite interromper a execução imediatamente com uma mensagem clara, evitando erros menos compreensíveis nas etapas seguintes.

In [0]:
source_path = Path(SOURCE_FILE_PATH)

if not source_path.is_file():
    raise FileNotFoundError(
        f"Arquivo {SOURCE_FILE_PATH} não encontrado: {SOURCE_FILE_PATH}"
    )

file_size_bytes = source_path.stat().st_size
file_size_mb = file_size_bytes / (1024 ** 2)
        
print("Arquivo de origem encontrado.")
print(f"Nome: {source_path.name}")
print(f"Tamanho: {file_size_mb:.2f} MB")
print(f"Caminho: {SOURCE_FILE_PATH}")

Arquivo de origem encontrado.
Nome: PLANILHA_GERAL_ABR_CAR.xlsx
Tamanho: 0.12 MB
Caminho: /Volumes/abr_car_dev/landing/source_files/PLANILHA_GERAL_ABR_CAR.xlsx


## 4. Preservação do snapshot semanal

O arquivo corrente da Landing pode ser atualizado pela loja. Antes da ingestão, preservamos uma cópia identificada pela data do lote.

Na primeira execução, o snapshot é criado. Em uma reexecução do mesmo lote, o arquivo existente é reutilizado e não é sobrescrito. A leitura da Bronze será realizada a partir desse snapshot preservado.

In [0]:
import shutil

snapshot_directory = Path(SNAPSHOT_DIRECTORY)
snapshot_path = Path(SNAPSHOT_FILE_PATH)

# Cria o diretório de snapshots caso ele ainda não exista
snapshot_directory.mkdir(parents=True, exist_ok=True)

if snapshot_path.exists():
    print("Snapshot já existente. A cópia preservada será reutilizada.")
else:
    shutil.copy2(source_path, snapshot_path)
    print("Snapshot criado com sucesso.")

# A Bronze lerá a cópia preservada deste lote
INGESTION_FILE_PATH = str(snapshot_path)

snapshot_size_mb = snapshot_path.stat().st_size / (1024 ** 2)

print(f"Arquivo de ingestão: {INGESTION_FILE_PATH}")
print(f"Tamanho do snapshot: {snapshot_size_mb:.2f} MB")
print(f"Batch ID: {BATCH_ID}")

Snapshot já existente. A cópia preservada será reutilizada.
Arquivo de ingestão: /Volumes/abr_car_dev/landing/source_files/snapshots/PLANILHA_GERAL_ABR_CAR_20260823.xlsx
Tamanho do snapshot: 0.12 MB
Batch ID: abr_car_20260823


## 5. Validação das abas do Excel

Antes da leitura dos dados, o processo confirma que todas as abas obrigatórias estão presentes no snapshot.

Se alguma aba estiver ausente, a execução será interrompida para impedir uma ingestão Bronze incompleta.

In [0]:
import pandas as pd

EXPECTED_SHEETS = [
    "Vendedores",
    "Veiculos",
    "Captacoes",
    "Custos_Veiculos",
    "Vendas",
    "Historico_Status",
    "Atendimentos_Procuras",
]

excel_file = pd.ExcelFile(
    INGESTION_FILE_PATH,
    engine="openpyxl",
)

available_sheets = excel_file.sheet_names

print("Abas encontradas:")
for sheet_name in available_sheets:
    print(f"- {sheet_name}")

missing_sheets = [
    sheet_name
    for sheet_name in EXPECTED_SHEETS
    if sheet_name not in available_sheets
]

extra_sheets = [
    sheet_name
    for sheet_name in available_sheets
    if sheet_name not in EXPECTED_SHEETS
]

if missing_sheets:
    raise ValueError(
        "O arquivo não contém todas as abas obrigatórias. "
        f"Abas ausentes: {missing_sheets}"
    )

print("\nTodas as sete abas obrigatórias foram encontradas.")

if extra_sheets:
    print(f"Abas adicionais identificadas: {extra_sheets}")
else:
    print("Nenhuma aba adicional foi identificada.")

Abas encontradas:
- Vendedores
- Veiculos
- Captacoes
- Custos_Veiculos
- Vendas
- Historico_Status
- Atendimentos_Procuras

Todas as sete abas obrigatórias foram encontradas.
Nenhuma aba adicional foi identificada.


## 6. Leitura das abas com Pandas

Cada aba do snapshot é carregada em um DataFrame Pandas independente.

Nesta etapa, os dados permanecem próximos à origem: não aplicamos correções, deduplicação ou regras de negócio. Inicialmente, validamos apenas a quantidade de linhas e colunas recebidas.

In [0]:
pandas_dfs = {}

for sheet_name in EXPECTED_SHEETS:
    pandas_dfs[sheet_name] = pd.read_excel(
        excel_file,
        sheet_name=sheet_name,
        engine="openpyxl",
    )

print("Dimensões das abas carregadas:\n")

for sheet_name, pandas_df in pandas_dfs.items():
    row_count, column_count = pandas_df.shape

    print(
        f"{sheet_name}: "
        f"{row_count} linhas x {column_count} colunas"
    )

Dimensões das abas carregadas:

Vendedores: 4 linhas x 7 colunas
Veiculos: 50 linhas x 18 colunas
Captacoes: 50 linhas x 12 colunas
Custos_Veiculos: 145 linhas x 10 colunas
Vendas: 34 linhas x 13 colunas
Historico_Status: 152 linhas x 6 colunas
Atendimentos_Procuras: 809 linhas x 13 colunas


## 7. Inspeção dos schemas inferidos pelo Pandas

Antes da conversão para Spark, inspecionamos os nomes das colunas e os tipos que o Pandas inferiu durante a leitura.

Essa inspeção é importante porque arquivos Excel não possuem um schema rígido. Uma mesma coluna pode conter números, textos, datas e células vazias, influenciando o tipo inferido.

In [0]:
for sheet_name, pandas_df in pandas_dfs.items():
    print(f"\n{'=' * 70}")
    print(f"Aba: {sheet_name}")
    print(f"{'=' * 70}")

    print(pandas_df.dtypes)


Aba: Vendedores
id_vendedor                     object
nome_vendedor                   object
data_admissao           datetime64[ns]
funcao                          object
status_vendedor                 object
email_corporativo               object
telefone_corporativo             int64
dtype: object

Aba: Veiculos
id_veiculo                         object
id_captacao                        object
placa                              object
chassi                             object
renavam                             int64
marca                              object
modelo                             object
versao                             object
ano_fabricacao                      int64
ano_modelo                          int64
cor                                object
combustivel                        object
cambio                             object
categoria                          object
quilometragem_entrada               int64
data_entrada               datetime64[ns]
preco_anu

## 8. Inspeção de valores nulos

Os valores ausentes são inspecionados antes da conversão para Spark porque podem alterar os tipos inferidos pelo Pandas.

Nesta camada, os nulos não serão preenchidos ou corrigidos. A Bronze deve registrar o dado recebido, enquanto a interpretação de obrigatoriedade e as regras de qualidade serão tratadas na Silver.

In [0]:
for sheet_name, pandas_df in pandas_dfs.items():
    null_counts = pandas_df.isna().sum()
    columns_with_nulls = null_counts[null_counts > 0]

    completely_empty_rows = pandas_df.isna().all(axis=1).sum()

    print(f"\nAba: {sheet_name}")
    print(f"Linhas completamente vazias: {completely_empty_rows}")

    if columns_with_nulls.empty:
        print("Nenhum valor nulo identificado.")
    else:
        print("Valores nulos por coluna:")
        print(columns_with_nulls)


Aba: Vendedores
Linhas completamente vazias: 0
Nenhum valor nulo identificado.

Aba: Veiculos
Linhas completamente vazias: 0
Nenhum valor nulo identificado.

Aba: Captacoes
Linhas completamente vazias: 0
Valores nulos por coluna:
id_venda_origem    40
dtype: int64

Aba: Custos_Veiculos
Linhas completamente vazias: 0
Nenhum valor nulo identificado.

Aba: Vendas
Linhas completamente vazias: 0
Nenhum valor nulo identificado.

Aba: Historico_Status
Linhas completamente vazias: 0
Nenhum valor nulo identificado.

Aba: Atendimentos_Procuras
Linhas completamente vazias: 0
Valores nulos por coluna:
valor_proposta    675
dtype: int64


## 9. Conversão dos DataFrames Pandas para Spark

O Pandas é utilizado para interpretar o arquivo Excel, pois oferece suporte adequado ao formato `.xlsx` e às suas abas.

Antes da gravação em Delta, os dados são convertidos para DataFrames Spark. O Spark integra-se ao Unity Catalog, permite gravação distribuída em Delta e será utilizado nas demais camadas da plataforma.

Os valores ausentes do Pandas são convertidos para `None`, permitindo que o Spark os represente como `null`.

In [0]:
spark_dfs = {}

for sheet_name, pandas_df in pandas_dfs.items():
    pandas_df_for_spark = (
        pandas_df
        .astype(object)
        .where(pd.notna(pandas_df), None)
    )

    spark_dfs[sheet_name] = spark.createDataFrame(
        pandas_df_for_spark
    )

    print(f"\nSchema Spark — {sheet_name}")
    spark_dfs[sheet_name].printSchema()


Schema Spark — Vendedores
root
 |-- id_vendedor: string (nullable = true)
 |-- nome_vendedor: string (nullable = true)
 |-- data_admissao: timestamp_ntz (nullable = true)
 |-- funcao: string (nullable = true)
 |-- status_vendedor: string (nullable = true)
 |-- email_corporativo: string (nullable = true)
 |-- telefone_corporativo: long (nullable = true)


Schema Spark — Veiculos
root
 |-- id_veiculo: string (nullable = true)
 |-- id_captacao: string (nullable = true)
 |-- placa: string (nullable = true)
 |-- chassi: string (nullable = true)
 |-- renavam: long (nullable = true)
 |-- marca: string (nullable = true)
 |-- modelo: string (nullable = true)
 |-- versao: string (nullable = true)
 |-- ano_fabricacao: long (nullable = true)
 |-- ano_modelo: long (nullable = true)
 |-- cor: string (nullable = true)
 |-- combustivel: string (nullable = true)
 |-- cambio: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- quilometragem_entrada: long (nullable = true)
 |-- data_e

## 10. Ajustes técnicos de identificadores

Colunas que representam identificadores devem ser armazenadas como texto, mesmo quando o Excel contém apenas dígitos.

Esse ajuste evita interpretações matemáticas indevidas e reduz o risco de perda de zeros à esquerda em arquivos futuros. Nenhum valor de negócio é corrigido ou preenchido nesta etapa.

In [0]:
from pyspark.sql import functions as F

spark_dfs["Vendedores"] = (
    spark_dfs["Vendedores"]
    .withColumn(
        "telefone_corporativo",
        F.col("telefone_corporativo").cast("string"),
    )
)

spark_dfs["Veiculos"] = (
    spark_dfs["Veiculos"]
    .withColumn(
        "renavam",
        F.col("renavam").cast("string"),
    )
)

print("Schema ajustado — Vendedores")
spark_dfs["Vendedores"].select(
    "telefone_corporativo"
).printSchema()

print("Schema ajustado — Veiculos")
spark_dfs["Veiculos"].select(
    "renavam"
).printSchema()

Schema ajustado — Vendedores
root
 |-- telefone_corporativo: string (nullable = true)

Schema ajustado — Veiculos
root
 |-- renavam: string (nullable = true)



## 11. Inclusão dos metadados técnicos

Cada registro da camada Bronze recebe metadados que permitem identificar sua origem e a execução responsável pela ingestão.

Essas informações possibilitam rastrear o registro até o arquivo, a aba e o lote semanal de origem, sem modificar os dados de negócio recebidos.

In [0]:
for sheet_name, spark_df in spark_dfs.items():
    spark_dfs[sheet_name] = (
        spark_df
        .withColumn(
            "_source_file",
            F.lit(SNAPSHOT_FILE_NAME),
        )
        .withColumn(
            "_source_sheet",
            F.lit(sheet_name),
        )
        .withColumn(
            "_batch_id",
            F.lit(BATCH_ID),
        )
        .withColumn(
            "_batch_date",
            F.lit(BATCH_DATE).cast("date"),
        )
        .withColumn(
            "_ingested_at",
            F.current_timestamp(),
        )
    )

print("Metadados adicionados aos sete DataFrames Spark.")

Metadados adicionados aos sete DataFrames Spark.


In [0]:
spark_dfs["Veiculos"].select(
    "_source_file",
    "_source_sheet",
    "_batch_id",
    "_batch_date",
    "_ingested_at",
).show(5, truncate=False)

+------------------------------------+-------------+----------------+-----------+--------------------------+
|_source_file                        |_source_sheet|_batch_id       |_batch_date|_ingested_at              |
+------------------------------------+-------------+----------------+-----------+--------------------------+
|PLANILHA_GERAL_ABR_CAR_20260823.xlsx|Veiculos     |abr_car_20260823|2026-08-23 |2026-08-24 04:07:56.581643|
|PLANILHA_GERAL_ABR_CAR_20260823.xlsx|Veiculos     |abr_car_20260823|2026-08-23 |2026-08-24 04:07:56.581643|
|PLANILHA_GERAL_ABR_CAR_20260823.xlsx|Veiculos     |abr_car_20260823|2026-08-23 |2026-08-24 04:07:56.581643|
|PLANILHA_GERAL_ABR_CAR_20260823.xlsx|Veiculos     |abr_car_20260823|2026-08-23 |2026-08-24 04:07:56.581643|
|PLANILHA_GERAL_ABR_CAR_20260823.xlsx|Veiculos     |abr_car_20260823|2026-08-23 |2026-08-24 04:07:56.581643|
+------------------------------------+-------------+----------------+-----------+--------------------------+
only showing top 5 

## 12. Validação das contagens após a conversão

A quantidade de registros de cada DataFrame Spark é comparada com a quantidade lida originalmente pelo Pandas.

A execução será interrompida caso alguma aba perca ou ganhe registros durante a conversão. As contagens não são fixadas manualmente, pois podem mudar nos próximos lotes semanais.

In [0]:
count_validation_results = []
count_mismatches = []

for sheet_name in EXPECTED_SHEETS:
    pandas_count = len(pandas_dfs[sheet_name])
    spark_count = spark_dfs[sheet_name].count()

    validation_status = (
        "OK"
        if pandas_count == spark_count
        else "DIVERGENTE"
    )

    count_validation_results.append(
        {
            "sheet_name": sheet_name,
            "pandas_count": pandas_count,
            "spark_count": spark_count,
            "status": validation_status,
        }
    )

    if pandas_count != spark_count:
        count_mismatches.append(
            {
                "sheet_name": sheet_name,
                "pandas_count": pandas_count,
                "spark_count": spark_count,
            }
        )

validation_df = spark.createDataFrame(
    count_validation_results
)

display(validation_df)

if count_mismatches:
    raise ValueError(
        "Divergências identificadas entre Pandas e Spark: "
        f"{count_mismatches}"
    )

print("Contagens validadas: nenhuma divergência identificada.")

pandas_count,sheet_name,spark_count,status
4,Vendedores,4,OK
50,Veiculos,50,OK
50,Captacoes,50,OK
145,Custos_Veiculos,145,OK
34,Vendas,34,OK
152,Historico_Status,152,OK
809,Atendimentos_Procuras,809,OK


Contagens validadas: nenhuma divergência identificada.


## 13. Gravação das tabelas Delta na camada Bronze

Os DataFrames Spark são gravados como tabelas Delta no Unity Catalog.

A gravação é idempotente por lote: se o mesmo `batch_id` for processado novamente, somente os registros desse lote serão substituídos. Os lotes históricos permanecerão preservados.

In [0]:
BRONZE_TABLE_NAMES = {
    "Vendedores": "vendedores",
    "Veiculos": "veiculos",
    "Captacoes": "captacoes",
    "Custos_Veiculos": "custos_veiculos",
    "Vendas": "vendas",
    "Historico_Status": "historico_status",
    "Atendimentos_Procuras": "atendimentos_procuras",
}

for sheet_name, table_name in BRONZE_TABLE_NAMES.items():
    full_table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    (
        spark_dfs[sheet_name]
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"_batch_id = '{BATCH_ID}'",
        )
        .saveAsTable(full_table_name)
    )

    print(
        f"Tabela gravada: {full_table_name}"
    )

Tabela gravada: abr_car_dev.bronze.vendedores
Tabela gravada: abr_car_dev.bronze.veiculos
Tabela gravada: abr_car_dev.bronze.captacoes
Tabela gravada: abr_car_dev.bronze.custos_veiculos
Tabela gravada: abr_car_dev.bronze.vendas
Tabela gravada: abr_car_dev.bronze.historico_status
Tabela gravada: abr_car_dev.bronze.atendimentos_procuras


## 14. Validação das tabelas Bronze persistidas

Após a gravação, as tabelas são lidas novamente diretamente do Unity Catalog.

A quantidade de registros do lote persistido é comparada com a quantidade preparada no Spark, confirmando que não houve perda ou duplicação durante a gravação Delta.

In [0]:
bronze_validation_results = []
bronze_mismatches = []

for sheet_name, table_name in BRONZE_TABLE_NAMES.items():
    full_table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    expected_count = spark_dfs[sheet_name].count()

    persisted_count = (
        spark.table(full_table_name)
        .filter(F.col("_batch_id") == BATCH_ID)
        .count()
    )

    validation_status = (
        "OK"
        if expected_count == persisted_count
        else "DIVERGENTE"
    )

    bronze_validation_results.append(
        {
            "table_name": full_table_name,
            "batch_id": BATCH_ID,
            "expected_count": expected_count,
            "persisted_count": persisted_count,
            "status": validation_status,
        }
    )

    if expected_count != persisted_count:
        bronze_mismatches.append(
            {
                "table_name": full_table_name,
                "expected_count": expected_count,
                "persisted_count": persisted_count,
            }
        )

bronze_validation_df = spark.createDataFrame(
    bronze_validation_results
)

display(bronze_validation_df)

if bronze_mismatches:
    raise ValueError(
        "Divergências identificadas nas tabelas Bronze: "
        f"{bronze_mismatches}"
    )

print(
    "Validação concluída: todas as tabelas Bronze "
    "foram persistidas corretamente."
)

batch_id,expected_count,persisted_count,status,table_name
abr_car_20260823,4,4,OK,abr_car_dev.bronze.vendedores
abr_car_20260823,50,50,OK,abr_car_dev.bronze.veiculos
abr_car_20260823,50,50,OK,abr_car_dev.bronze.captacoes
abr_car_20260823,145,145,OK,abr_car_dev.bronze.custos_veiculos
abr_car_20260823,34,34,OK,abr_car_dev.bronze.vendas
abr_car_20260823,152,152,OK,abr_car_dev.bronze.historico_status
abr_car_20260823,809,809,OK,abr_car_dev.bronze.atendimentos_procuras


Validação concluída: todas as tabelas Bronze foram persistidas corretamente.


## 15. Registro da auditoria da ingestão

O resultado da validação é armazenado no schema `ops`, criando um histórico técnico dos lotes processados.

A tabela de auditoria permite verificar quais tabelas foram processadas, quantos registros eram esperados, quantos foram persistidos e o status final de cada validação.

In [0]:
AUDIT_TABLE = f"{CATALOG}.ops.bronze_ingestion_audit"

bronze_audit_df = (
    bronze_validation_df
    .withColumn(
        "source_file",
        F.lit(SNAPSHOT_FILE_NAME),
    )
    .withColumn(
        "batch_date",
        F.lit(BATCH_DATE).cast("date"),
    )
    .withColumn(
        "validated_at",
        F.current_timestamp(),
    )
)

(
    bronze_audit_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "replaceWhere",
        f"batch_id = '{BATCH_ID}'",
    )
    .saveAsTable(AUDIT_TABLE)
)

print(f"Auditoria gravada: {AUDIT_TABLE}")

Auditoria gravada: abr_car_dev.ops.bronze_ingestion_audit


In [0]:
display(
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
    .orderBy("table_name")
)

batch_id,expected_count,persisted_count,status,table_name,source_file,batch_date,validated_at
abr_car_20260823,809,809,OK,abr_car_dev.bronze.atendimentos_procuras,PLANILHA_GERAL_ABR_CAR_20260823.xlsx,2026-08-23,2026-08-24T04:08:28.211Z
abr_car_20260823,50,50,OK,abr_car_dev.bronze.captacoes,PLANILHA_GERAL_ABR_CAR_20260823.xlsx,2026-08-23,2026-08-24T04:08:28.211Z
abr_car_20260823,145,145,OK,abr_car_dev.bronze.custos_veiculos,PLANILHA_GERAL_ABR_CAR_20260823.xlsx,2026-08-23,2026-08-24T04:08:28.211Z
abr_car_20260823,152,152,OK,abr_car_dev.bronze.historico_status,PLANILHA_GERAL_ABR_CAR_20260823.xlsx,2026-08-23,2026-08-24T04:08:28.211Z
abr_car_20260823,50,50,OK,abr_car_dev.bronze.veiculos,PLANILHA_GERAL_ABR_CAR_20260823.xlsx,2026-08-23,2026-08-24T04:08:28.211Z
abr_car_20260823,34,34,OK,abr_car_dev.bronze.vendas,PLANILHA_GERAL_ABR_CAR_20260823.xlsx,2026-08-23,2026-08-24T04:08:28.211Z
abr_car_20260823,4,4,OK,abr_car_dev.bronze.vendedores,PLANILHA_GERAL_ABR_CAR_20260823.xlsx,2026-08-23,2026-08-24T04:08:28.211Z


In [0]:
display(
    spark.sql(
        f"""
        SELECT
            table_name,
            batch_id,
            expected_count,
            persisted_count,
            status,
            validated_at
        FROM {AUDIT_TABLE}
        WHERE batch_id = '{BATCH_ID}'
        ORDER BY table_name
        """
    )
)

table_name,batch_id,expected_count,persisted_count,status,validated_at
abr_car_dev.bronze.atendimentos_procuras,abr_car_20260823,809,809,OK,2026-08-24T04:08:28.211Z
abr_car_dev.bronze.captacoes,abr_car_20260823,50,50,OK,2026-08-24T04:08:28.211Z
abr_car_dev.bronze.custos_veiculos,abr_car_20260823,145,145,OK,2026-08-24T04:08:28.211Z
abr_car_dev.bronze.historico_status,abr_car_20260823,152,152,OK,2026-08-24T04:08:28.211Z
abr_car_dev.bronze.veiculos,abr_car_20260823,50,50,OK,2026-08-24T04:08:28.211Z
abr_car_dev.bronze.vendas,abr_car_20260823,34,34,OK,2026-08-24T04:08:28.211Z
abr_car_dev.bronze.vendedores,abr_car_20260823,4,4,OK,2026-08-24T04:08:28.211Z
